# AMPidentifier — Google Colab / Jupyter Notebook Guide

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/madsondeluna/AMPidentifier/blob/main/AMPidentifier_Colab_Guide.ipynb)

**AMPidentifier** predicts antimicrobial peptides (AMPs) from amino acid sequences in FASTA format. It computes 22 physicochemical and compositional descriptors and classifies each peptide using one of six trained models: Random Forest, SVM, Gradient Boosting, XGBoost, LightGBM, or a soft-voting ensemble of all five.

- PyPI: https://pypi.org/project/ampidentifier
- Web interface: https://ampidentifier.madsondeluna.com
- GitHub: https://github.com/madsondeluna/AMPidentifier
- Reference: Luna-Aragão et al. (2026). *AMPidentifier: A Cross-Platform Ensemble Toolkit for Antimicrobial Peptide Prediction.*

## 1. Installation

Install the package via pip. This installs all Python dependencies and the `ampidentifier` CLI entry point.

The trained model files (`.pkl`) are stored in the GitHub repository and must be downloaded separately because of their size. The cell below handles both steps.

In [ ]:
# Install the AMPidentifier package from PyPI
!pip install -q ampidentifier

In [ ]:
import os

# Download only the model files from the repository (sparse checkout)
if not os.path.exists("model_training/tuned_model"):
    !git clone --filter=blob:none --no-checkout https://github.com/madsondeluna/AMPidentifier.git _amp_repo
    !cd _amp_repo && git sparse-checkout init --cone && \
     git sparse-checkout set model_training/tuned_model && \
     git checkout main
    !cp -r _amp_repo/model_training .
    !rm -rf _amp_repo
    print("Model files downloaded.")
else:
    print("Model files already present.")

In [ ]:
# Verify installation
!ampidentifier --help

## 2. Preparing the input FASTA file

The input must be a valid FASTA file. Rules:
- Each entry starts with a `>` header line containing a unique identifier.
- The sequence line contains standard one-letter amino acid codes (case-insensitive).
- Minimum sequence length: 5 residues.
- Accepted characters: `A C D E F G H I K L M N P Q R S T V W Y B X Z U O J * -`

You can also upload your own `.fasta` file using the file browser in the left panel (Colab) or place it in the working directory (Jupyter).

In [ ]:
fasta_content = """\
>Magainin-2|Xenopus_laevis|Cationic_amphipathic_helix
GIGKFLHSAKKFGKAFVGEIMNS
>LL-37|Homo_sapiens|Cathelicidin_family
LLGDFFRKSKEKIGKEFKRIVQRIKDFLRNLVPRTES
>Melittin|Apis_mellifera|Venom_peptide
GIGAVLKVLTTGLPALISWIKRKRQQ
>Defensin-alpha-1|Homo_sapiens|Defensin_family
ACYCRIPACIAGERRYGTCIYQGRLWAFCC
>Insulin_Chain_B|Homo_sapiens|Peptide_hormone
FVNQHLCGSHLVEALYLVCGERGFFYTPKT
>Glucagon|Homo_sapiens|Peptide_hormone
HSQGTFTSDYSKYLDSRRAQDFVQWLMNT
>Vasoactive_intestinal_peptide|Homo_sapiens|Neuropeptide
HSDAVFTDNYTRLRKQMAVKKYLNSILN
>Substance_P|Homo_sapiens|Neuropeptide
RPKPQQFFGLM
"""

with open("input.fasta", "w") as f:
    f.write(fasta_content)

print(f"input.fasta created with {fasta_content.count('>')} sequences.")

## 3. CLI reference

| Flag | Short | Type | Required | Default | Description |
|---|---|---|---|---|---|
| `--input` | `-i` | path | Yes | — | Path to input FASTA file |
| `--output_dir` | `-o` | path | Yes | — | Output directory (created if absent) |
| `--model` | `-m` | choice | No | `voting` | Model to use (see options below) |
| `--threshold` | — | float 0–1 | No | MCC-optimized | Decision threshold for AMP classification |

**Available models (`-m`):**

| Value | Full name | Accuracy | AUC-ROC | MCC |
|---|---|---|---|---|
| `voting` | Soft-voting ensemble (recommended) | 92.9% | 0.977 | 0.859 |
| `lgbm` | LightGBM | 92.7% | 0.975 | 0.855 |
| `xgb` | XGBoost | 92.2% | 0.974 | 0.843 |
| `gb` | Gradient Boosting | 92.0% | 0.974 | 0.839 |
| `rf` | Random Forest | 91.9% | 0.972 | 0.839 |
| `svm` | Support Vector Machine (RBF kernel) | 91.9% | 0.969 | 0.839 |

## 4. Basic usage — voting ensemble (default)

When `-m` is omitted, the soft-voting ensemble is used. This is the recommended configuration.

In [ ]:
!ampidentifier -i input.fasta -o output_voting/

## 5. Selecting a specific model with `-m`

In [ ]:
!ampidentifier -i input.fasta -o output_lgbm/ -m lgbm

In [ ]:
!ampidentifier -i input.fasta -o output_rf/ -m rf

In [ ]:
!ampidentifier -i input.fasta -o output_svm/ -m svm

In [ ]:
!ampidentifier -i input.fasta -o output_xgb/ -m xgb

In [ ]:
!ampidentifier -i input.fasta -o output_gb/ -m gb

## 6. Custom decision threshold with `--threshold`

By default, each model uses its MCC-optimized threshold from the validation set. Pass `--threshold` to override it.

- **Lower value** (e.g. `0.30`): higher sensitivity, more AMP predictions, more false positives.
- **Higher value** (e.g. `0.70`): higher specificity, fewer AMP predictions, fewer false positives.

Adjust when downstream cost of false positives and false negatives differs (e.g. shortlisting candidates for wet-lab synthesis).

In [ ]:
# High-specificity run: only high-confidence AMP predictions
!ampidentifier -i input.fasta -o output_highspec/ -m voting --threshold 0.70

In [ ]:
# High-sensitivity run: broader classification
!ampidentifier -i input.fasta -o output_highsens/ -m voting --threshold 0.30

## 7. Reading and interpreting the output

Each run produces `predictions_{model}.csv` in the output directory:

| Column | Type | Description |
|---|---|---|
| `ID` | string | Sequence identifier from the FASTA header |
| `sequence` | string | Amino acid sequence |
| `probability_AMP` | float | Predicted probability of being an AMP (0.0–1.0) |
| `prediction` | int | Binary label: `1` = AMP, `0` = non-AMP |
| `label` | string | Human-readable: `AMP` or `non-AMP` |

In [ ]:
import pandas as pd

df = pd.read_csv("output_voting/predictions_voting.csv")
df

In [ ]:
# Filter and rank predicted AMPs by probability
amps = df[df["prediction"] == 1].sort_values("probability_AMP", ascending=False)
print(f"{len(amps)} AMP(s) predicted out of {len(df)} input sequences.")
amps[["ID", "probability_AMP", "label"]]

## 8. Visualizing results

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

df_sorted = df.sort_values("probability_AMP", ascending=True)
colors = ["#059669" if p == 1 else "#dc2626" for p in df_sorted["prediction"]]

fig, ax = plt.subplots(figsize=(9, max(3, len(df_sorted) * 0.45)))
ax.barh(df_sorted["ID"], df_sorted["probability_AMP"], color=colors, edgecolor="none", height=0.6)
ax.set_xlabel("Probability AMP", fontsize=10)
ax.set_xlim(0, 1)
ax.spines[["top", "right"]].set_visible(False)
ax.legend(
    handles=[mpatches.Patch(color="#059669", label="AMP"), mpatches.Patch(color="#dc2626", label="non-AMP")],
    frameon=False, fontsize=9
)
plt.tight_layout()
plt.savefig("amp_predictions.png", dpi=150, bbox_inches="tight")
plt.show()

## 9. Batch comparison — all models at once

In [ ]:
import os

models = ["voting", "lgbm", "xgb", "gb", "rf", "svm"]

for m in models:
    os.makedirs(f"output_{m}", exist_ok=True)
    os.system(f"ampidentifier -i input.fasta -o output_{m}/ -m {m}")

In [ ]:
import pandas as pd

models = ["voting", "lgbm", "xgb", "gb", "rf", "svm"]
dfs = []

for m in models:
    tmp = pd.read_csv(f"output_{m}/predictions_{m}.csv")[["ID", "probability_AMP", "prediction"]]
    tmp = tmp.rename(columns={"probability_AMP": f"prob_{m}", "prediction": f"pred_{m}"})
    dfs.append(tmp.set_index("ID"))

comparison = pd.concat(dfs, axis=1)
comparison

## 10. Python API — programmatic usage

Call `run_prediction_pipeline` directly from Python for integration into larger workflows.

In [ ]:
from amp_identifier.core import run_prediction_pipeline

run_prediction_pipeline(
    input_file="input.fasta",
    output_dir="output_api/",
    internal_model_type="voting",  # options: "rf", "svm", "gb", "xgb", "lgbm", "voting"
    use_ensemble=False,
)

In [ ]:
import pandas as pd

result = pd.read_csv("output_api/predictions_voting.csv")
result

## 11. Tips

**Recommended model:** Use `voting` for best overall performance (AUC-ROC 0.977, MCC 0.859 on internal test; AUC-ROC 0.950, MCC 0.742 on the independent benchmark n=4,736).

**Threshold selection:** Default thresholds are MCC-optimized. For high-throughput screening where false positives are costly, increase the threshold (e.g. `--threshold 0.65`). For candidate discovery where sensitivity matters, decrease it (e.g. `--threshold 0.35`).

**Sequence length:** The training set contains AMPs in the 5–50 residue range. Longer sequences can be submitted, but predictions for residues outside this range may be less reliable.

**Partial AMP activity:** Some non-AMP proteins contain sequence regions with physicochemical properties typical of AMPs (cationic and amphipathic). A non-AMP label refers to the full sequence, not to individual domains or fragments.

**Uploading your FASTA in Colab:** Use the file browser on the left panel (`Files` icon) to upload a `.fasta` file, then update `-i` to point to it.